# CrossSkinNet: এক জায়গায় পুরো workflow

PAD+HAM development, PH2 external; DermNet optional এবং default-এ বন্ধ। PAD, HAM10000 ও PH2 Add Input দিয়ে attach করুন। Image copy চিনে 429টি decision দিতে হবে না: paper config অনিশ্চিত development groups আলাদা cohort exclusion হিসেবে রাখে। এটি duplicate verdict নয়। কী বাদ গেল তা report-এ থাকবে। Real training-এর জন্য GPU এবং weights/install-এর জন্য Internet On করুন।

In [ ]:
from pathlib import Path
import subprocess, sys, os

repo=Path("/kaggle/working/Codex")
if not repo.exists():
    subprocess.run(["git","clone","https://github.com/mahabubhasanmahin/Codex.git",str(repo)],check=True)
elif (repo/".git").exists():
    subprocess.run(["git","pull","--ff-only","origin","main"],cwd=repo,check=True)
else:
    raise RuntimeError("Existing Codex folder is not a Git checkout; preserve it before continuing")
os.chdir(repo)
subprocess.run([sys.executable,"-m","pip","install","-r","requirements.txt"],check=True)


## 1. একবার data প্রস্তুত করুন
Source images বদলাবে না। Config-এর source paths actual mount-এর সঙ্গে মিলতে হবে। নতুন unseen field/site পাওয়া গেলে code error দিয়ে সঠিক প্রয়োজন জানাবে।

In [ ]:
subprocess.run([sys.executable,"scripts/08_paper.py","prepare","--config","configs/paper.yaml"],check=True)


## 2. প্রকৃত cohort report দেখুন
Class/group যথেষ্ট না থাকলে prepare থামবে। Count না দেখে training শুরু করবেন না।

In [ ]:
import yaml, json, csv
cfg=yaml.safe_load(Path("configs/paper.yaml").read_text())
output=Path(cfg["paths"]["output_root"])
print((output/"COHORT_REPORT.md").read_text())
with (output/"cohort_flow.csv").open() as handle:
    for row in csv.DictReader(handle):
        print(row)


## 3. একটি model train করুন
প্রথম real experiment: EfficientNet CrossSkinNet, fine-tuned, seed 42। Epoch schedule config থেকে আসবে। প্রয়োজন হলে পরের session-এ একই command দিয়ে resume হবে; পুরো output folder সংরক্ষণ করতে হবে।

In [ ]:
BACKBONE = "efficientnet"  # efficientnet / swin / dinov2 / dinov3
ARCHITECTURE = "crossskinnet"  # crossskinnet / image_only / concat
TRAINING_SEED = 42  # 42, 43, 44; prepared split stays fixed
FROZEN = False
experiment_args = ["--backbone", BACKBONE, "--architecture", ARCHITECTURE,
                   "--seed", str(TRAINING_SEED), "--frozen" if FROZEN else "--no-frozen"]
subprocess.run([sys.executable,"scripts/08_paper.py","train","--config","configs/paper.yaml",*experiment_args],check=True)


## 4. একই experiment evaluate করুন
Final evaluation-এ time-budget-stopped incomplete training গ্রহণ হবে না; আগে resume করে schedule/early stopping সম্পূর্ণ করুন। PH2 কোনো tuning/calibration-এ যাবে না।

In [ ]:
subprocess.run([sys.executable,"scripts/08_paper.py","evaluate","--config","configs/paper.yaml",*experiment_args],check=True)
print("Results:",output/"results/paper_results.csv")


## পরের experiments
শুধু plan দেখাবে, সবগুলো একসঙ্গে চালাবে না। একই prepared split রাখতে হবে। Architecture `image_only` বা `concat` baseline; backbones efficientnet/swin/dinov2/dinov3; frozen flag এবং seed বদলে পৃথক runs। DINOv3-এর access/license/HF_TOKEN প্রয়োজন হতে পারে।

In [ ]:
subprocess.run([sys.executable,"scripts/08_paper.py","plan","--config","configs/paper.yaml"],check=True)


## সীমা
এই notebook code execution সহজ করে; paper acceptance বা real performance নিশ্চয়তা দেয় না। Quarantine selection bias ও class/source/group exclusion counts paper-এ লিখতে হবে। Full protocol ও command variations: `docs/PAPER_WORKFLOW.md`। Tiny synthetic results thesis result নয়।